In [9]:
from data_frame.analytical.window_function.window_definer import WindowDefiner
from data_frame.analytical.window_function.moving_analyzer import MovingAnalyzer
from data_frame.spark_utils import get_spark
import datetime
from pyspark.sql import functions as F

In [5]:
spark = get_spark(app_name="Analytical functions")

In [8]:
# Create time series data

time_data = [
    (datetime.date(2024, 1, 1), 100),
    (datetime.date(2024, 1, 2), 150),
    (datetime.date(2024, 1, 3), 120),
    (datetime.date(2024, 1, 4), 180),
    (datetime.date(2024, 1, 5), 200),
    (datetime.date(2024, 1, 6), 170),
    (datetime.date(2024, 1, 7), 190)
]
df_time = spark.createDataFrame(time_data, ["date", "value"])

## 1. Moving Averages and Sums

In [7]:

# Define moving window (3-day moving average)
moving_spec = WindowDefiner.define_moving_window(
    partition_cols=[],  # No partition
    order_cols=["date"],
    preceding=2,  # Look back 2 rows
    following=0   # Current row only
)

# Calculate moving average
df_analyzed = MovingAnalyzer.moving_average(
    df_time, moving_spec, "value", 3, "3day_moving_avg"
)

print("Moving average (3-day):")
df_analyzed.show()

Moving average (3-day):


26/03/18 12:26:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:26:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:26:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:26:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:26:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----+------------------+
|      date|value|   3day_moving_avg|
+----------+-----+------------------+
|2024-01-01|  100|             100.0|
|2024-01-02|  150|             125.0|
|2024-01-03|  120|123.33333333333333|
|2024-01-04|  180|             150.0|
|2024-01-05|  200|166.66666666666666|
|2024-01-06|  170|183.33333333333334|
|2024-01-07|  190|186.66666666666666|
+----------+-----+------------------+



## 2. Lag and Lead Analysis

In [10]:
# Define ordered window for lag/lead
order_spec = WindowDefiner.define_ordered_window(
    partition_cols=[],
    order_cols=["date"],
    order_direction="asc"
)

# Calculate lag and lead
df_lagged = MovingAnalyzer.lag_analysis(
    df_time, order_spec, "value", 1, "previous_day"
)
df_lagged = MovingAnalyzer.lead_analysis(
    df_lagged, order_spec, "value", 1, "next_day"
)

# Calculate day-over-day change
df_lagged = df_lagged.withColumn(
    "daily_change", 
    F.col("value") - F.col("previous_day")
)

print("Lag/Lead analysis:")
df_lagged.show()

Lag/Lead analysis:


26/03/18 12:28:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:28:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:28:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:28:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:28:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----+------------+--------+------------+
|      date|value|previous_day|next_day|daily_change|
+----------+-----+------------+--------+------------+
|2024-01-01|  100|        NULL|     150|        NULL|
|2024-01-02|  150|         100|     120|          50|
|2024-01-03|  120|         150|     180|         -30|
|2024-01-04|  180|         120|     200|          60|
|2024-01-05|  200|         180|     170|          20|
|2024-01-06|  170|         200|     190|         -30|
|2024-01-07|  190|         170|    NULL|          20|
+----------+-----+------------+--------+------------+



## 3. First and Last Values

In [11]:
# Define unbounded window
unbounded_spec = WindowDefiner.define_unbounded_window(
    partition_cols=[],
    order_cols=["date"]
)

# Get first and last values in window
df_first_last = MovingAnalyzer.first_last_value(
    df_time, unbounded_spec, "value", 
    "first_value_in_period", "last_value_in_period"
)

print("First and last values in entire dataset:")
df_first_last.show()

First and last values in entire dataset:


26/03/18 12:28:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:28:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:28:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:28:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:28:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----+---------------------+--------------------+
|      date|value|first_value_in_period|last_value_in_period|
+----------+-----+---------------------+--------------------+
|2024-01-01|  100|                  100|                 190|
|2024-01-02|  150|                  100|                 190|
|2024-01-03|  120|                  100|                 190|
|2024-01-04|  180|                  100|                 190|
|2024-01-05|  200|                  100|                 190|
|2024-01-06|  170|                  100|                 190|
|2024-01-07|  190|                  100|                 190|
+----------+-----+---------------------+--------------------+



## 4. Cumulative Distribution

In [12]:
# Calculate cumulative distribution
df_cdf = MovingAnalyzer.cumulative_distribution(
    df_time, order_spec, "value", "cdf"
)

print("Cumulative distribution:")
df_cdf.orderBy("value").show()

Cumulative distribution:


26/03/18 12:29:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:29:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:29:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:29:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 12:29:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----+-------------------+
|      date|value|                cdf|
+----------+-----+-------------------+
|2024-01-01|  100|0.14285714285714285|
|2024-01-03|  120|0.42857142857142855|
|2024-01-02|  150| 0.2857142857142857|
|2024-01-06|  170| 0.8571428571428571|
|2024-01-04|  180| 0.5714285714285714|
|2024-01-07|  190|                1.0|
|2024-01-05|  200| 0.7142857142857143|
+----------+-----+-------------------+

